# Jiaqi's papers

Collaging all of Jiaqi's papers and collating them into a pdf book.

In [1]:
import os
import requests
import pandas as pd
import time
import re
import sqlite3

## Fetch metadata from ORCID

Use a few ORCID APIs to get the details of the papers.  Thanks copilot!
It creates a csv file [papers.csv](papers.csv) and an sqlite3 database [papers.db](papers.db)

In [2]:
from helpers import fetch_orcid_papers

# --------------------------
# Configuration
# --------------------------
ORCID_ID = "0000-0001-6491-3851"
CROSSREF_CONTACT_EMAIL = "your_email@leeds.ac.uk"
USE_CROSSREF = True

if os.path.exists("papers.csv"):
    print("papers.csv already exists — skipping ORCID download.")
else:
    print("papers.csv not found — fetching from ORCID...")
    df = fetch_orcid_papers(ORCID_ID, CROSSREF_CONTACT_EMAIL, USE_CROSSREF)
    df.to_csv("papers.csv", index=False)
    print("\nDone. Saved papers.csv")

papers.csv already exists — skipping ORCID download.


## Download papers (that are available publicly)

In [39]:
import json

df = pd.read_csv("papers.csv")
os.makedirs("pdfs", exist_ok=True)

for _, row in df.iterrows():
    doi = row["doi"]
    print(f"DOI: {doi}")
    if pd.isna(doi):
        continue

    
    api = f"https://api.unpaywall.org/v2/{doi}?email=n.s.malleson@leeds.ac.uk"
    resp = requests.get(api, timeout=20)

    try:
        r = resp.json()
    except json.JSONDecodeError:
        print(f"\tUnpaywall returned invalid JSON for {doi}. Status code {resp.status_code}. Skipping.")
        #print("\t",resp.text[:200])
        continue

    oa = r.get("best_oa_location") or {}
    pdf_url = oa.get("url_for_pdf")


    if pdf_url:
        pdf_data = requests.get(pdf_url).content
        filename = f"pdfs/{doi.replace('/', '_')}.pdf"
        with open(filename, "wb") as f:
            f.write(pdf_data)
        print("\tDownloaded", filename)
    else:
        print("\tNot available")


DOI: 10.1061/jtepbs.teeng-9272
	Not available
DOI: 10.1371/journal.pstr.0000162
	Downloaded pdfs/10.1371_journal.pstr.0000162.pdf
DOI: 10.25937/8bf3-h968
	Unpaywall returned invalid JSON for 10.25937/8bf3-h968. Skipping.
	 404
	 <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try agai
DOI: 10.17632/9v4byyvgxh.1
	Unpaywall returned invalid JSON for 10.17632/9v4byyvgxh.1. Skipping.
	 404
	 <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try agai
DOI: 10.4230/LIPIcs.GIScience.2023.78
	Unpaywall returned invalid JSON for 10.4230/LIPIcs.GIScience.2023.78. Skipping.
	 404
	 <!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you e

## Re-read and arrange the papers

Read the csv file created above (necessary if the script has been re-run without re-downloading everything).

Order them by ascending year then month and do some other things (like make integers from floats)

In [2]:
df = pd.read_csv("papers.csv")

#Sort ascending date
df = df.sort_values(by=["year", "month"], ascending=True)

# Convert to ints
for x in ["year", "month", "day", "volume", "issue"]:
    df[x] = df[x].astype("Int64")

Do some manual fine tuning (removing duplicates etc.)

In [3]:
dois_to_drop = []

# Remove this arxiv version of a paper that has been published in a journal ('Agent-based modelling for Urban Analytics ..')
dois_to_drop.append("10.48550/arxiv.2210.06955")

# Loads of version of 'A simulation model of pedestrian flow based on Bayesian Nash Equilibrium and a Multi-Agent System'. 
# Just keep the mains ones
dois_to_drop.append("10.5281/zenodo.6411582")
dois_to_drop.append("10.5281/zenodo.7825536")
dois_to_drop.append("10.25937/75wf-aa82")
dois_to_drop.append("10.48550/arXiv.2211.14260")
dois_to_drop.append("10.5281/zenodo.7825536")
dois_to_drop.append("10.25937/8bf3-h968")
#dois_to_drop.append("10.4230/LIPIcs.GIScience.2023.78")

# OSF preprint of "Social influence and meat-eating" (keep journal version 10.3390/su14137935)
dois_to_drop.append("10.31219/osf.io/u2tmd")

# CrimRxiv preprint of "Learning the rational choice perspective" (keep journal version)
dois_to_drop.append("10.21428/cb6ab371.18137891")

# Dataset, not a paper ("Experimental data of evacuation simulations")
dois_to_drop.append("10.17632/9v4byyvgxh.1")


for doi_to_drop in dois_to_drop:
    df = df.loc[df.doi != doi_to_drop,]

## Find missing abstracts

In [4]:
# Add abstracts. Claude found most of these for me

abstract = "The lack of experimental datasets for individual behaviours has hindered the systematic studies of pedestrian behaviours as well as the refined development of regular laws of individual movement in simulation models. This research developed a simulation model for crowd evacuation on the basis of Bayesian Nash Equilibrium (BNE) and a Multi-Agent System (MAS). BNE was introduced in this research to augment the rationality of individual decision-making process in evacuation simulation and to assist pedestrians in discovering an optimal evacuation route to avoid congestions. A series of simulation experiments were conducted to evaluate the performance of the initial model, and the current experimental results demonstrate a noticeable positive influence of BNE on reducing evacuation time. A detailed introduction of the establishment and implementation details of the model as well as model analysis have been provided in this paper. Limitations and a few optional research directions in the future are also discussed."
df.loc[df.doi=="10.4230/LIPIcs.GIScience.2023.78","abstract"]=abstract

# An Agent-Based Simulation Model of Pedestrian Evacuation Based on Bayesian Nash Equilibrium (JASSS 2023)
abstract = "This research incorporates Bayesian game theory into pedestrian evacuation in an agent-based model. Three pedestrian behaviours were compared: Random Follow, Shortest Route and Bayesian Nash Equilibrium (BNE), as well as combinations of these. The results showed that BNE pedestrians were able to evacuate more quickly as they predict congestion levels in their next step and adjust their directions to avoid congestion, closely matching the behaviours of evacuating pedestrians in reality. A series of simulation experiments were conducted to evaluate whether and how BNE affects pedestrian evacuation procedures. The results showed that: 1) BNE has a large impact on reducing evacuation time; 2) BNE pedestrians displayed more intelligent and efficient evacuating behaviours; 3) As the proportion of BNE users rises, average evacuation time decreases, and average comfort level increases. A detailed description of the model and relevant experimental results is provided in this paper. Several limitations as well as further works are also identified."
df.loc[df.doi=="10.18564/jasss.5037","abstract"]=abstract

# Calibrating Agent-Based Models Using Uncertainty Quantification Methods (JASSS 2022)
abstract = "Agent-based models (ABMs) can be found across a number of diverse application areas ranging from simulating consumer behaviour to infectious disease modelling. Part of their popularity is due to their ability to simulate individual behaviours and decisions over space and time. However, whilst there are plentiful examples within the academic literature, these models are only beginning to make an impact within policy areas. Whilst frameworks such as NetLogo make the creation of ABMs relatively easy, a number of key methodological issues, including the quantification of uncertainty, remain. In this paper we draw on state-of-the-art approaches from the fields of uncertainty quantification and model optimisation to describe a novel framework for the calibration of ABMs using History Matching and Approximate Bayesian Computation. The utility of the framework is demonstrated on three example models of increasing complexity: (i) Sugarscape to illustrate the approach on a toy example; (ii) a model of the movement of birds to explore the efficacy of our framework and compare it to alternative calibration approaches and; (iii) the RISC model of farmer decision making to demonstrate its value in a real application. The results highlight the efficiency and accuracy with which this approach can be used to calibrate ABMs. This method can readily be applied to local or national-scale ABMs, such as those linked to the creation or tailoring of key policy decisions."
df.loc[df.doi=="10.18564/jasss.4791","abstract"]=abstract

# Exploring the Combined Effect of Factors Influencing Commuting Patterns and CO2 Emissions (JASSS 2016)
abstract = "This paper develops an agent-based model of the daily commute in Aberdeen City and the surrounding area in Scotland, UK. We study the impact of flexitime work arrangements, urban concentration, a new bypass, and cycle lanes on commute time length, reliability and CO2 emissions, and analyse the diverse conflation of these factors and the different connections of them in order to detect their cumulative effects. Our results suggest that flexitime will reduce CO2 emissions from traffic. It also reduces mean commute time and makes commute time more reliable. We find that although higher urban concentration will make travel time less reliable, it will reduce CO2 emissions from commuting and cut commute time length. There might also be a trade-off between travel time length and reliability regarding urban concentration. We show that the new bypass will only reduce mean commute time by a small amount, while slightly increasing total CO2 emissions. Finally, we find that cyclists sharing roads with cars do not necessarily slow down the traffic on the whole. We conclude that infrastructural, social and urban issues should never be studied in isolation with each other, and that urban policies will have ramifications for both urban and surrounding ex-urban areas."
df.loc[df.doi=="10.18564/jasss.3078","abstract"]=abstract

# Simulating the Actions of Commuters Using a Multi-Agent System (JASSS 2019)
abstract = "The activity of commuting to and from a place of work affects not only those travelling but also wider society through their contribution to congestion and pollution. It is desirable to have a means of simulating commuting in order to allow organisations to predict the effects of changes to working patterns and locations and inform decision making. In this paper, we outline an agent-based software framework that combines real-world data from multiple sources to simulate the actions of commuters. We demonstrate the framework using data supplied by an employer based in the City of Edinburgh UK. We demonstrate that the BDI-inspired decision-making framework used is capable of forecasting the transportation modes to be used. Finally, we present a case study, demonstrating the use of the framework to predict the impact of moving staff within the organisation to a new work site."
df.loc[df.doi=="10.18564/jasss.4007","abstract"]=abstract

# The ODD Protocol (JASSS 2020)
abstract = "The Overview, Design concepts and Details (ODD) protocol for describing Individual- and Agent-Based Models (ABMs) is now widely accepted and used to document such models in journal articles. As a standardized document for providing a consistent, logical and readable account of the structure and dynamics of ABMs, some research groups also find it useful as a workflow for model design. Even so, there are still limitations to ODD that obstruct its more widespread adoption. Such limitations are discussed and addressed in this paper: the limited availability of guidance on how to use ODD; the length of ODD documents; limitations of ODD for highly complex models; lack of sufficient details of many ODDs to enable reimplementation without access to the model code; and the lack of provision for sections in the document structure covering model design rationale, the model's underlying narrative, and the means by which the model's fitness for purpose is evaluated. We document the steps we have taken to provide better guidance on: structuring complex ODDs and an ODD summary for inclusion in a journal article (with full details in supplementary material; Table 1); using ODD to point readers to relevant sections of the model code; update the document structure to include sections on model rationale and evaluation. We also further advocate the need for standard descriptions of simulation experiments and argue that ODD can in principle be used for any type of simulation model. Thereby ODD would provide a lingua franca for simulation modelling."
df.loc[df.doi=="10.18564/jasss.4259","abstract"]=abstract

# From oil wealth to green growth (Environmental Modelling & Software 2018)
abstract = "This paper develops an empirical, multi-layered and spatially-explicit agent-based model that explores sustainable pathways for Aberdeen city and surrounding area to transition from an oil-based economy to green growth. The model takes an integrated, complex systems approach to urban systems and incorporates the interconnectedness between people, households, businesses, industries and neighbourhoods. We find that the oil price collapse could potentially lead to enduring regional decline and recession. With green growth, however, the crisis could be used as an opportunity to restructure the regional economy, reshape its neighbourhoods, and redefine its identity in the global economy. We find that the type of the green growth and the location of the new businesses will have profound ramifications for development outcomes, not only by directly creating businesses and employment opportunities in strategic areas, but also by redirecting households and service businesses to these areas. New residential and business centres emerge as a result of this process. Finally, we argue that industries, businesses and the labour market are essential components of a deeply integrated urban system. To understand urban transition, models should consider both household and industrial aspects."
df.loc[df.doi=="10.1016/j.envsoft.2018.05.017","abstract"]=abstract

# Endogenous rise and collapse of housing price (CEUS 2017)
abstract = "On the basis of interviews with local real estate agents, this study develops an agent-based model of housing market to determine the cause of rise and collapse of US housing price during the years immediately preceding the US financial crisis (2007-2009). We study the key factors affecting housing price volatility, such as lenient financing and speculation. The dynamic simulation findings in the study show in concrete terms how lenient lending practices combined with speculation can lead to increased volatility in housing price, including sharp rises immediately followed by collapses. The exploratory work in this study will contribute to the understanding of the causes of housing bubbles and inform policy decisions."
df.loc[df.doi=="10.1016/j.compenvurbsys.2016.11.005","abstract"]=abstract

# Learning the rational choice perspective (CEUS 2024)
abstract = "Over the past 15 years, environmental criminologists have explored the application of agent-based models (ABMs) of crime events and various theoretical frameworks applied to understand them. Models have supported criminological theorising and, in some cases, been applied to make predictions about the impact of interventions devised to reduce crime. However, decision-making frameworks utilised in criminological ABMs have typically been implemented through traditional techniques such as condition-action rules. While these models have provided significant insights, they neglect a crucial component of theoretical accounts of offending, the notion that offenders are learning agents whose behavioural dynamics change over time and space. In response, this article presents an ABM of residential burglary in which offender agents utilise reinforcement learning (RL) to learn behaviours. This solution enables offender agents to learn from individual-level perceptions of the environment and, given these perceptions, develop behavioural responses that benefit themselves. The model includes conceptualisations of the Routine Activity Theory (RAT), Crime Pattern Theory (CPT) and a utility function, Target Attractiveness, which acts as a behavioural mould to nudge offender agents to learn behaviours in keeping with the Rational Choice Perspective (RCP). Trained behaviours are then tested by introducing crime prevention interventions into the model and examining the reactions of offender agents. In keeping with empirical studies of offending, experimental results demonstrate that offender agents utilising RL learn to offend at targets where rewards outweigh risks and effort, offend close to home, frequently victimise high-rewarding targets, and conversely learn to avoid offending in areas associated with high levels of risk and effort."
df.loc[df.doi=="10.1016/j.compenvurbsys.2024.102141","abstract"]=abstract

# Mitigating housing market shocks (Journal of Simulation 2024)
abstract = "Research in modelling housing market dynamics using agent-based models (ABMs) has grown due to the rise of accessible individual-level data. This research involves forecasting house prices, analysing urban regeneration, and the impact of economic shocks. There is a trend towards using machine learning (ML) algorithms to enhance ABM decision-making frameworks. This study investigates exogenous shocks to the UK housing market and integrates reinforcement learning (RL) to adapt housing market dynamics in an ABM. Results show agents can learn real-time trends and make decisions to manage shocks, achieving goals like adjusting the median house price without pre-determined rules. This model is transferable to other housing markets with similar complexities. The RL agent adjusts mortgage interest rates based on market conditions. Importantly, our model shows how a central bank agent learned conservative behaviours in sensitive scenarios, aligning with a 2009 study, demonstrating emergent behavioural patterns."
df.loc[df.doi=="10.1080/17477778.2024.2375446","abstract"]=abstract

# Too much of a good thing? (Journal of Transport Geography 2018)
abstract = "Using a spatial agent-based model of transport, this paper explores various \"unconventional\" workplace sharing programmes that allow employees to work remotely at other work sites in Northeast Scotland, with Aberdeenshire Council as the focal employer. We attempt to answer the following research questions: (i) To what extent do systemic effects arising from agent interactions within the transport network mitigate or enhance any potential benefits of workplace sharing? (ii) How are these effects changed by informal workplace practices influenced by organizational structure and corporate culture, as opposed to formal business policy? We have been able to show that there are potential benefits to workplace sharing, particularly within a large organization with spatially distributed workplaces. Indeed, the greater the flexibility available, the larger the potential gains, especially with participation of the whole workforce across all employers. However, the apparent benefits of workplace sharing for commuting times and CO2 emissions from transport can be negated by organizational structure and corporate culture. Informal policies whereby team leaders stipulate collocation of team members to facilitate day-to-day and face-to-face interaction can even lead to a worse situation than the case where there is no workplace sharing. The effect of the sharing programmes also depends on the spatial distribution of existing road network, as well as industrial and residential areas. The work acts as a warning that apparently attractive \"win-win\" policies with the potential to promote better staff welfare, reduce pollution and make more efficient use of infrastructure can be negated by informal practices in workplaces. It is a step towards a general policy simulation platform where the effectiveness of transport policies can be tested and potential unintended consequences detected before they are implemented in reality, by which time it may be too late or costly to correct any unintended negative effects."
df.loc[df.doi=="10.1016/j.jtrangeo.2018.04.005","abstract"]=abstract

# Modelling food security (Global Environmental Change 2021)
abstract = "Achieving food and nutrition security for all in a changing and globalized world remains a critical challenge of utmost importance. The development of solutions benefits from insights derived from modelling and simulating the complex interactions of the agri-food system, which range from global to household scales and transcend disciplinary boundaries. A wide range of models based on various methodologies (from food trade equilibrium to agent-based) seek to integrate direct and indirect drivers of change in land use, environment and socio-economic conditions at different scales. However, modelling such interaction poses fundamental challenges, especially for representing non-linear dynamics and adaptive behaviours. We identify key pieces of the fragmented landscape of food security modelling, and organize achievements and gaps into different contextual domains of food security (production, trade, and consumption) at different spatial scales. Building on in-depth reflection on three core issues of food security -- volatility, technology, and transformation -- we identify methodological challenges and promising strategies for advancement. We emphasize particular requirements related to the multifaceted and multiscale nature of food security. They include the explicit representation of transient dynamics to allow for path dependency and irreversible consequences, and of household heterogeneity to incorporate inequality issues. To illustrate ways forward we provide good practice examples using meta-modelling techniques, non-equilibrium approaches and behavioural-based modelling endeavours. We argue that further integration of different model types is required to better account for both multi-level agency and cross-scale feedbacks within the food system."
df.loc[df.doi=="10.1016/j.gloenvcha.2020.102085","abstract"]=abstract

# From primary data to formalized decision-making (Ecology and Society 2024)
abstract = "Model-based analyses effectively investigate leverage points for sustainability transformations in agriculture by systematically evaluating policies under varying environmental, economic, or institutional conditions. Agent-based modeling proves particularly valuable for analyzing agricultural systems because it represents individual farmers -- key actors in land use -- their interactions, and resulting landscape-level patterns. However, formalizing empirically observed farmer behavior into model rules presents challenges, especially with qualitative data. This article guides modelers through formalizing farmer decision-making based on empirical findings in three steps: selecting approach, choosing key elements, and translating findings. The authors illustrate their framework using literature examples and their own model representing farmer decision-making regarding agri-environmental scheme adoption across Europe. By systematizing this process, agent-based models become less stylized, offering greater potential for supporting policymakers in identifying sustainable agricultural transformation strategies."
df.loc[df.doi=="10.5751/ES-15400-290431","abstract"]=abstract

# Crossing the chasm (GeoInformatica 2019)
abstract = "Agent-based models (ABMs) simulate actions and interactions of autonomous agents and groups and their effect on systems as a whole, accounting for learning without assuming perfect rationality or complete knowledge. As such, they are well-suited to the study of complex adaptive socio-environmental systems, in which there is a need to explore the consequences of policy decisions in a context where the people whose behaviour is affected by those decisions are also able to respond and adapt to them. There has been a move away from abstract, theory-driven ABMs towards more empirically-grounded applications that stakeholders find more credible and relevant, though this introduces further challenges around validation and system description. Using Geoffrey Moore's Crossing the Chasm as a lens, we argue that the way ahead for ABM lies in identifying the niches in which it can best demonstrate its advantages, working with collaborators to show that it can deliver on its promises. We conclude by highlighting several areas requiring further development to establish ABM as an expected methodology for exploring complex socio-environmental policy scenarios in spatially-distributed systems."
df.loc[df.doi=="10.1007/s10707-018-00340-z","abstract"]=abstract

# Simulating Urban Transition in Major Socio-Economic Shocks (WSC 2021)
abstract = "This paper presents an agent-based model that simulates the dynamic process of urban transition in major socio-economic shocks. The model examines coupled housing and labour markets across diverse neighbourhoods in a city undergoing industrial restructuring, where traditional industries are replaced by new ones. The research makes three key contributions: first, it establishes connections between housing and labour markets by linking individual and business choices to neighbourhood evolution; second, it links local workers and businesses to broader industrial structure changes within the urban system; and third, it proposes a complex systems framework for analyzing major socio-economic transitions in urban environments. Preliminary scenario analysis results from the simulation are presented."
df.loc[df.doi=="10.1109/wsc52266.2021.9715510","abstract"]=abstract

# An Agent-Based Model of UK Farmers' Decision-Making on Adoption of Agri-environment Schemes (ESSA 2022)
abstract = "Agri-environment schemes (AES) are government-funded voluntary programs that incentivise farmers and land managers for environmentally friendly farming practices. Understanding farmers' decision-making process and its impact on AES adoption can aid policy makers in designing AES schemes that meet adoption goals and environmental targets. This paper presents a spatially explicit agent-based model (ABM) -- BESTMAP-ABM-UK -- that simulates farmers' decision-making process, inclusive of farmers' social, behavioural and economic factors, on adopting buffer strips, cover crops, grassland management and arable land conversion to grassland schemes in the UK. The model produces farmers' AES adoption under varied AES scheme designs in terms of the contract length, the offered payment level, the bureaucracy level and the required minimal area."
df.loc[df.doi=="10.1007/978-3-031-34920-1_37","abstract"]=abstract

# A Preliminary Study of Individual Based Crowd Simulation Based on Bayesian Nash Equilibrium (ESSA 2022)
abstract = "The lack of experimental datasets for individual behaviours has hindered the systematic studies of pedestrian behaviours as well as the refined development of regular laws of individual movement in simulation models. This research developed a simulation model for crowd evacuation on the basis of Bayesian Nash Equilibrium (BNE) and a Multi-Agent System (MAS). BNE was introduced in this research to augment the rationality of individual decision-making process in evacuation simulation and to assist pedestrians in discovering an optimal evacuation route to avoid congestions. A series of simulation experiments were conducted to evaluate the performance of the initial model, and the current experimental results demonstrate a noticeable positive influence of BNE on reducing evacuation time."
df.loc[df.doi=="10.1007/978-3-031-34920-1_26","abstract"]=abstract

# -----------------------------------------------------------------------
# Papers with missing abstracts (not freely accessible online):
# - 10.1016/j.ecolecon.2017.09.006 — Does Work-life Balance Affect Pro-environmental Behaviour?
# - 10.1016/j.energy.2018.09.078 — Do renters skimp on energy efficiency during economic recessions?
# - 10.1016/j.enpol.2017.05.025 — Exploring factors affecting on-farm renewable energy adoption in Scotland
# - 10.1007/978-3-642-54783-6_10 — Who Creates Housing Bubbles? An Agent-Based Study
# - 10.1017/9781108553827.003 — Agent-based Models of Coupled Social and Natural Systems
# - 10.1061/jtepbs.teeng-9272 — Investigating the Role of Station Staff Guidance on Pedestrian Flows
# - 10.1136/jech-2022-ssmabstracts.7 — Causal inference (conference abstract)
# - 10.1007/s11403-022-00375-4 — Introduction to the special issue on agent-based models in urban economics (editorial)
# -----------------------------------------------------------------------

Some more that I found myself

In [5]:
#df.loc[df.doi=="10.1061/jtepbs.teeng-9272"]
#df.loc[df.title=="Agent-based Models of Coupled Social and Natural Systems",:]

In [6]:
abstract = "This paper uses large-scale micro data to identify key factors affecting the decision to adopt renewable energy generation (wind, solar and biomass) on farms in Scotland. We construct an integrated dataset that includes the compulsory agricultural census and farm structural survey that cover almost all farms in Scotland. In addition to farm owner demographics and farm business structures, we also assess the effect of diversification activities such as tourism and forestry, as well as the spatial, biophysical and geophysical attributes of the farms on the adoption decision. We find that diversified farms are more likely to adopt renewable energy, especially solar and biomass energy. Farms are also more likely to adopt renewable energy if they have high local demand for energy, or suitable conditions for renewable energy production. We find that biophysical factors such as the agricultural potential of farm land are important in adoption decisions. We identify adopter profiles for each type of renewable energy, and map the geographic location of potential adopters. We argue that renewable energy policy should be more integrated with farm diversification policy and farm support schemes. It should also be tailored for each type of renewable energy, for the potential adopter profiles of wind, solar and biomass energy all differ in farm characteristics and geographic distribution."
df.loc[df.doi=="10.1016/j.enpol.2017.05.025","abstract"]=abstract

abstract = "The environmental challenges we face today have made the need to behave pro-environmentally increasingly salient. Many believe that the modern day “busyness” of life and lack of spare time have kept people from acting according to their values and behaving more pro-environmentally. This study uses microdata from the UK Household Longitudinal Study (UKHLS) to investigate the relation between pro-environmental behaviour, environmental self-perception and work-life balance. Pro-environmental behaviour covers 21 behaviours relating to home energy, personal transport, recycling and shopping. Work-life balance is defined with relation to the availability of discretionary time using both objective and subjective measures. The results from the regression models of overall pro-environmental behaviour suggest that work-life imbalance does not appear to affect, neither directly nor indirectly through environmental values and attitudes, pro-environmental behaviour. The main factors determining the extent of pro-environmental behaviour relate to individual's attitudes towards the environment, age, educational attainment, household income and the presence of young children. The sensitivity analysis looking at differing time demanding behaviours reveals that actual availability of discretionary time does not seem to affect pro-environmental behaviour, while the subjective experience of work-life imbalance can have a negative direct effect particularly for more time demanding pro-environmental behaviours."
df.loc[df.doi=="10.1016/j.ecolecon.2017.09.006","abstract"]=abstract

abstract = "This paper investigates tenants' willingness to pay (WTP) for energy efficiency in the private rented housing sector. Using data from Aberdeen city and Shire in Scotland between the third quarter of 2013 and the second quarter of 2017, rent premiums of 2–11% associated with more energy efficient dwellings are found, and the magnitudes of these premiums are considerable compared to those of other physical attributes. Such premiums however, are significantly reduced during economic recession, suggesting that tenants' WTP for energy efficiency varies under different economic conditions. From a methodological perspective, the study uses a multilevel model, where the unobservable neighbourhood and age effects are approximated. Our results implicate that although tenants’ WTP for more energy efficient is present, there still might be a need for public strategy to facilitate the improvement of energy performance in the private rented sector."
df.loc[df.doi=="10.1016/j.energy.2018.09.078","abstract"]=abstract

abstract = "Agent-based models are dynamic computer simulations that explicitly represent the interactions of heterogeneous individuals. Interest in such models stems from a number of disciplines. Some economists see agent-based models as enabling them to escape the restrictive assumptions of human rationality needed for tractable mathematical analysis under the classical paradigm, among other reasons (Axtell, 2000). Indeed, the broad affiliation of disciplines interested in a ‘complex systems’ perspective, in which systems of multiple interacting heterogeneous elements generate ‘emergent’ structure and order at the aggregate scale, offers a new metaphor for understanding economic systems. Arthur, Durlauf and Lane’s (1997) introduction to The Economy as a Complex Evolving System, for example, cites various features of real economic systems that are challenging to classical analysis, but entirely natural from a complex systems perspective: e.g. out-of-equilibrium dynamics, dispersed interaction and the lack of a global mediator. Agent-based models are closely aligned conceptually to a complex systems view of the world. Broader interest in agent-based modelling in the social sciences is derived from its perceived potential as a ‘third way’ between the quantitative and qualitative camps (Moss, 1999). The conceptual chasm between these two is often overemphasised, with most pragmatic social scientists willing to adopt mixed-methods approaches to case studies, but if seen as a formal environment in which to explore the dynamic outcomes of more assumptions than the human mind can reason with logically, agent-based models offer qualitative social scientists new tools to explore their findings, which can potentially be fitted to data gathered and analysed by quantitative social scientists. Geographers are interested in agent-based models because they can be used to represent space explicitly."
df.loc[df.title=="Agent-based Models of Coupled Social and Natural Systems","abstract"] = abstract

abstract = "Background Understanding which factors predict or cause individuals’ dropout from weight-loss programmes can provide insight into possible adaptations that minimise dropout. Prior research indicating ‘predictors’ of dropout does not allow causal interpretation, which limits its utility to inform possible adaptations. Thus, this study examines dropout in weight-loss programmes through the re-analysis of secondary datasets to: (1) explore if, and by how much, insights differ between prediction and causal inference approaches; and (2) identify associated methodological issues in both prediction and causal inference. Methods We used data from eight systematically identified studies published between 2013–2020 on factors related to dropout from weight-loss programmes. The eight primary analyses attempted to identify variables that predict or are correlated with whether a participant dropped out of a weight-loss intervention, without necessarily recognising that this might conflate the two distinct data science tasks of prediction and causal inference. The same datasets were re-analysed for prediction and causal inference, using approaches appropriate for each task. To identify predictors, a single model containing a full set of variables was developed for each dataset. To estimate causal effects, a directed acyclic graph (DAG) was coproduced with authors from each original study, and multiple models were generated depending on the exposures of interest. The results (and corresponding conclusions) were then compared. Results In the primary analyses, age and body mass index (BMI) (with inconsistency in the direction of the relationship) were often identified as predictors of dropout, while in the secondary analyses, factors causally associated with dropout were older age, women, higher baseline weight, and more challenging weight-loss targets. None of the primary studies could inform causal interpretation due to methodological issues such as Table 2 Fallacy, reversal paradox, and challenges related to analysing composite variables. With the exception of two studies, all causal analyses suffered positivity violations. Conclusion Weight-loss programme design cannot be improved by knowing which factors predict dropout; in fact, this may cause more harm than good. There were some differences in the coefficients generated by the prediction and causal inference models (even sign reversal), indicating that the potential for misinterpreting causal relationships could be severe. To identify causal factors related to dropout, a causal inference evaluation, informed by an appropriate DAG, is preferred, but existing study datasets may not meet the criteria for robust inference. Our findings suggest that study designs appropriate for causal analysis should be considered when determining which weight-loss programme adaptations are needed to reduce dropout."
df.loc[df.doi=="https://doi.org/10.1136/jech-2022-ssmabstracts.7","abstract"] = abstract

abstract = "Exploring the critical roles of various factors on determining the wayfinding performance of passengers during a subway fire emergency is important. By developing a virtual reality environment of an underground urban railway station, an immersive virtual reality escape experiment was conducted with a head-mounted device to record the wayfinding performance of 1,151 participants. The experimental data include trajectory data of the participants in the escape task (e.g., time and three-dimensional coordinates) and survey data of personal information. Binary logistic regression, general linear regression, LightGBM, and other methods were employed to analyze the effects of station staffs’ guidance, emergency broadcast, signage, and other factors on wayfinding performance, with station staffs’ guidance being a factor rarely addressed in previous studies. The large sample size enabled the application of ensemble learning methods and facilitated fine-grained group difference analyses that are difficult to achieve in traditional small-sample evacuation experiments. Results show that emergency broadcast and the guidance of station staffs had a highly significant impact on evacuees’ route choice. Education exhibited a positive influence on both evacuees’ route choices and evacuation time. Furthermore, information overload was observed among evacuees under the influence of multiple information sources. The results of this study can not only serve as a theoretical basis for policymakers to understand the determinants of wayfinding performance, but also provide guidance for underground urban railway station or metro station evacuation plan in a fire emergency."
df.loc[df.doi=="10.1061/jtepbs.teeng-9272","abstract"] = abstract

## Other cleaning

In [7]:
# Generic DOI dedup: for any remaining duplicate DOIs, prefer entries without Python list-style authors
def has_bracket_authors(row):
    return isinstance(row.authors, str) and row.authors.startswith("[")

df["_bracket_authors"] = df.apply(has_bracket_authors, axis=1)
df = df.sort_values("_bracket_authors")  # False first (good entries first)
df = df.drop_duplicates(subset="doi", keep="first")
df = df.drop(columns=["_bracket_authors"])
df = df.sort_values(by=["year", "month"], ascending=True)

In [8]:
#df.to_csv("~/Desktop/temp.csv")

## Create latex

There is a template file called [papers_template.tex](papers_template.tex). Now populate it with the papers. 
This is slightly more complicated than I thought it would be at first because copilot was quite careful with formatting the references.

In [9]:
from helpers import write_chapters_tex

write_chapters_tex(df, "chapters.tex")

Wrote chapters.tex
